In [ ]:
import re
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

# ========================
# Funções auxiliares
# ========================
def normalize_text(s):
    """Normaliza texto (sem acentos, minúsculo, sem caracteres estranhos)."""
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r'[^a-z0-9x ]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def extract_numbers_and_units(s):
    """Extrai números e checa se têm unidade (ml, mg, g, ui)."""
    if pd.isna(s):
        return []
    s = str(s).lower()
    matches = re.findall(r'(\d+(?:[\.,]\d+)?)\s*(ml|mg|g|ui)?', s)
    result = []
    for num, unit in matches:
        num = float(num.replace(',', '.'))
        result.append((num, bool(unit)))
    return result

def choose_candidate_pack(text):
    """Escolhe melhor número como pack-size, ignorando unidades."""
    nums = extract_numbers_and_units(text)
    candidates = [int(round(n)) for n, has_unit in nums if not has_unit and 2 <= n <= 500]
    if not candidates:
        return 0
    return max(candidates)

def find_times_pattern(s):
    """Procura padrões tipo '3x50' ou '3 x 50'."""
    if pd.isna(s):
        return None
    s = str(s).lower()
    m = re.search(r'(\d{1,4})\s*[x×]\s*(\d{1,4})', s)
    if m:
        return int(m.group(1)), int(m.group(2))
    return None

def heuristic_multiplier(material, nomeconc):
    """Heurística final para decidir multiplicador e pack-size."""
    m_pack = choose_candidate_pack(material)
    n_pack = choose_candidate_pack(nomeconc)

    m_times = find_times_pattern(material)
    n_times = find_times_pattern(nomeconc)

    multiplier = 1.0
    pack = max(m_pack, n_pack, 0)

    if m_times:
        multiplier = m_times[0]
        pack = max(pack, m_times[1])
    elif n_times:
        multiplier = n_times[0]
        pack = max(pack, n_times[1])
    else:
        if n_pack and not m_pack:
            multiplier = 1.0 / n_pack
        elif m_pack and not n_pack:
            multiplier = 1.0
    return multiplier, pack

# ========================
# Carregar dados
# ========================
tabela = pd.read_excel("Banco_de_Fardos.xlsx")
tabela.columns = tabela.columns.str.strip()

# Normalizar texto
tabela["MATERIAL"] = tabela["MATERIAL"].apply(normalize_text)
tabela["NOME_CONC"] = tabela["NOME_CONC"].apply(normalize_text)

# ========================
# Heurísticas aplicadas na base
# ========================
tabela["multiplier"], tabela["pack"] = zip(*tabela.apply(lambda row: heuristic_multiplier(row["MATERIAL"], row["NOME_CONC"]), axis=1))

# ========================
# Treino IA (para complementar regras)
# ========================
X = tabela[["MATERIAL", "NOME_CONC"]]
y_conv = tabela["CONVERSAO"]
y_fator = tabela["FATOR"]

# Codificação
le_mat = LabelEncoder()
le_nome = LabelEncoder()
X_enc = pd.DataFrame({
    "MATERIAL": le_mat.fit_transform(X["MATERIAL"]),
    "NOME_CONC": le_nome.fit_transform(X["NOME_CONC"])
})

modelo_conv = RandomForestClassifier(random_state=42)
modelo_fator = RandomForestRegressor(random_state=42)

modelo_conv.fit(X_enc, y_conv)
modelo_fator.fit(X_enc, y_fator)

# ========================
# Previsão em novos dados
# ========================
novos = tabela.copy()
X_novos = pd.DataFrame({
    "MATERIAL": le_mat.transform(novos["MATERIAL"]),
    "NOME_CONC": le_nome.transform(novos["NOME_CONC"])
})

novos["PREV_CONV"] = modelo_conv.predict(X_novos)
novos["PREV_FATOR"] = modelo_fator.predict(X_novos).round(2)

# ========================
# Ajuste final com heurísticas
# ========================
novos["AJUSTADO_CONV"] = novos["PREV_CONV"]
novos["AJUSTADO_FATOR"] = novos["PREV_FATOR"]

for i, row in novos.iterrows():
    mult, pack = heuristic_multiplier(row["MATERIAL"], row["NOME_CONC"])
    if mult != 1.0 or pack > 0:
        novos.at[i, "AJUSTADO_CONV"] = "multiplica" if mult > 1 else "divide" if mult < 1 else row["PREV_CONV"]
        novos.at[i, "AJUSTADO_FATOR"] = mult if mult != 1.0 else row["PREV_FATOR"]

# ========================
# Salvar resultado
# ========================
saida = "PlanilhaAtualizada.xlsx"
novos.to_excel(saida, index=False)
print(f"✅ Resultado salvo em: {saida}")
